# Stage 1 — Obtain the data

This notebook downloads every series used in the project from the U.S. Energy Information
Administration (EIA), saves each one as a CSV named after the metric it measures, and
writes `data/raw/manifest.csv`, which records the coverage of each file.

The two kinds of data come from **two different EIA products**:

| Data | Source | Why |
|---|---|---|
| 11 weekly petroleum series | `.xls` history files | complete, no key needed |
| 2 daily spot prices (WTI, Brent) | **EIA API v2** | the published `.xls` price files silently skip hundreds of ordinary trading days |

| Folder | Content | In Git? |
|---|---|---|
| `data/raw/xls/` | original EIA `.xls` downloads | no |
| `data/raw/weekly/` | weekly petroleum data as CSV, e.g. `crude_stocks.csv` (public domain) | yes |
| `data/raw/prices/` | daily WTI and Brent prices as CSV (third-party data, see README) | no |

Files that are already present are not downloaded again, so the notebook can be re-run
quickly.

**The API needs a free key.** It is read from `EIA_API_KEY` in a file called `.env` in the
repository root, which is in `.gitignore`. The key is never printed by this notebook. See
the README for how to create it.

*Written with the help of Claude (Anthropic); declared in the paper appendix.*

In [ ]:
import sys
from pathlib import Path

# Find the repository root (the folder containing src/) so `src` can be imported
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import (MANIFEST_PATH, MIN_DAILY_PRICES_PER_YEAR, SAMPLE_END,
                        SAMPLE_START, WEEKS_PER_YEAR)
from src.data_utils import (PRICE_NAMES, api_route_metadata, api_series_names,
                            build_manifest, compare_price_sources, convert_xls_to_csv,
                            download_all, download_prices, observations_per_year,
                            read_series, series_table, weekly_observations_per_year)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("Repository root:", ROOT)

## 1. The series used

In [ ]:
series_table()

## 2. Check the API key and confirm the series codes

The request below is the smallest one the API offers: the metadata of the spot price
route. It confirms three things — that the key in `.env` is valid, that the route has a
`daily` frequency, and that the facet used to pick a series is called `series`.

Nothing here prints the key. If this cell fails with a message about the key, create or
correct `.env` as described in the README.

In [ ]:
meta = api_route_metadata()
print("route period range:", meta["startPeriod"], "to", meta["endPeriod"])
print("frequencies available:", [f["id"] for f in meta["frequency"]])
print("facets available:", [f["id"] for f in meta["facets"]])
print("\nThe API key works.")

In [ ]:
# The two codes the project needs, as the API itself describes them
names = api_series_names()
names[names["series_id"].isin(["RWTC", "RBRTE"])]

## 3. Download the original EIA `.xls` files

Each file is about 100–500 KB. If a download fails, the reason is shown in the `status`
column and the other files are still downloaded. These files provide the 11 weekly series;
the two price files are downloaded as well, but only to check the API against them.

In [ ]:
status = download_all()
status

## 4. Convert the weekly `.xls` files to CSV

Each `.xls` file is recognised from the EIA code stored inside it and saved as
`date, <metric>`. The two price files are deliberately skipped here — they are replaced by
the API download in the next step.

In [ ]:
converted = convert_xls_to_csv()
converted

## 5. Download the daily prices from the EIA API

The API returns at most 5000 rows per request, so each series is fetched in pages with an
increasing `offset`, sorted by date ascending.

The published `.xls` price files are still used for two checks: the prices must be
identical on every date both sources cover, and the few days that are present only in the
`.xls` are added back, so that neither source's gaps survive.

In [ ]:
prices_report = download_prices()
prices_report

## 6. Checks

### 6.1 Daily prices per year

A full year of trading has about 250 days. Any year below 240 is listed: only the first
year of each series (partial history) and the current year (incomplete) should appear.

In [ ]:
counts = pd.concat([observations_per_year(name) for name in PRICE_NAMES], axis=1)
print("daily prices per year:")
display(counts.T)

low = counts[(counts < MIN_DAILY_PRICES_PER_YEAR).any(axis=1)]
print(f"years with fewer than {MIN_DAILY_PRICES_PER_YEAR} observations:")
display(low if not low.empty else "none")

### 6.2 The API against the published `.xls` files

`price_differs` counts the dates where the two sources give a different price; it should be
0, or very close to it. `added_by_api` counts the trading days the `.xls` file was missing,
and `only_in_xls` the few days the API is missing, which are filled from the `.xls`.

In [ ]:
for name in PRICE_NAMES:
    table = compare_price_sources(name)
    if table.empty:
        print(f"{name}: the .xls file is not present, so no comparison is possible\n")
        continue
    totals = table[["added_by_api", "only_in_xls", "price_differs"]].sum()
    print(f"=== {name.upper()} ===")
    print(f"days added by the API      : {totals['added_by_api']}")
    print(f"days only in the .xls file : {totals['only_in_xls']} (filled back in)")
    print(f"dates where prices differ  : {totals['price_differs']}")
    worst = table[table["added_by_api"] > 0].nlargest(5, "added_by_api")
    print("the five years the .xls file was missing most days:")
    display(worst)

### 6.3 The weekly EIA series

Each weekly series should have 52 or 53 observations per year. Years with fewer are listed:
only the first year of each series, the current year, and the early 1980s (when EIA
published less often) should appear.

In [ ]:
weekly_counts = weekly_observations_per_year()
gaps = weekly_counts[(weekly_counts < WEEKS_PER_YEAR).any(axis=1)]
print(f"years where a weekly series has fewer than {WEEKS_PER_YEAR} observations:")
display(gaps.fillna(0).astype(int))

# the sample years come from config; the final part-year is left out
first_year = pd.Timestamp(SAMPLE_START).year
last_full_year = pd.Timestamp(SAMPLE_END).year - 1
in_sample = weekly_counts.loc[first_year:last_full_year]
odd = in_sample[(in_sample.notna() & (in_sample < WEEKS_PER_YEAR)).any(axis=1)]
print(f"same check restricted to {first_year}-{last_full_year} (the sample, "
      "excluding the final part-year):")
display(odd.fillna(0).astype(int) if not odd.empty
        else f"none - every series has {WEEKS_PER_YEAR}-{WEEKS_PER_YEAR + 1} weeks every year")

## 7. Check the files and write the manifest

`data/raw/manifest.csv` is the **data version record**. The price CSVs cannot be committed
(third-party data), so the manifest is what lets anyone confirm they are using the same
data: for every series it stores the source, when it was downloaded (UTC), the number of
rows, the first and last date, and a **SHA-256 hash** of the CSV file. The hash identifies
the file without containing any of it, so the manifest is safe to commit.

Notebook 02 re-hashes the price files and warns if they no longer match this record.

In [ ]:
manifest = build_manifest()
manifest[["metric", "frequency", "source", "status", "first_date", "last_date", "n_obs", "n_missing"]]

### The version record

In [ ]:
record = manifest[["metric", "source", "downloaded_utc", "n_rows",
                   "first_date", "last_date", "sha256"]].copy()
# show only the start of each hash so the table stays readable; the CSV holds it in full
record["sha256"] = record["sha256"].str.slice(0, 16) + "..."
record

In [ ]:
problems = manifest[manifest["status"] != "ok"]
if not problems.empty:
    raise RuntimeError(
        "These series are missing or unreadable:\n"
        + problems[["metric", "status", "url"]].to_string(index=False)
    )
print(f"All {len(manifest)} metrics are available as CSV.")
print("Manifest saved to", MANIFEST_PATH.relative_to(ROOT))

## 8. Sanity check

The WTI spot price went negative once, in April 2020. The cell below locates that day from
the prices themselves and shows the days around it, which confirms the price series was
downloaded correctly. It is dealt with in stage 2.

In [ ]:
# The non-positive price is found from the data, not written in: this shows the
# days around whichever day the prices themselves say settled below zero.
main_series = PRICE_NAMES[0]
wti = read_series(main_series)
negative = wti[wti <= 0]
print(f"{len(negative)} non-positive {main_series} price(s) found: "
      f"{[str(d.date()) for d in negative.index]}")
centre = negative.index[0]
wti.loc[centre - pd.Timedelta(days=3):centre + pd.Timedelta(days=2)]

**Next:** `02_clean_data.ipynb` turns these files into one weekly table aligned to EIA
release dates.